In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

# ======== Parameters ========
data_dir = '/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/Patch Classification/Sorted_data_split/train'
batch_size = 32
num_epochs = 300
learning_rate = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======== Transforms (NO Resize Needed if already 64x64) ========
transform = transforms.Compose([
    transforms.Resize((64, 64)),   # <-- IMPORTANT
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ======== Dataset & DataLoader ========
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
num_classes = len(dataset.classes)

train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)

# ======== Modified ResNet18 for 64x64 ========
model = models.resnet18(pretrained=True)

# Adjust first layer for small images
model.conv1 = nn.Conv2d(
    3, 64,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False
)

# Remove maxpool
model.maxpool = nn.Identity()

# Replace classifier
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

# ======== Loss & Optimizer ========
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# ======== Training Loop ========
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(num_epochs):

    # ---- Train ----
    model.train()
    running_loss, correct, total = 0, 0, 0

    prog_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, labels in prog_bar:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        prog_bar.set_postfix(loss=loss.item())

    train_loss = running_loss / total
    train_acc  = correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # ---- Validation ----
    model.eval()
    running_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc  = correct / total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}  |  Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}  |  Val Acc:   {val_acc:.4f}")

# ======== Save Model ========
torch.save(model, '/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/Patch Classification/resnet18_full_coconut.pth')
print("\nTraining Complete ✅")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 140MB/s]
Epoch 1/300 [Train]: 100%|██████████| 8/8 [01:41<00:00, 12.73s/it, loss=0.735]



Epoch [1/300]
Train Loss: 1.8246  |  Train Acc: 0.5000
Val Loss:   2.5480  |  Val Acc:   0.1967


Epoch 2/300 [Train]: 100%|██████████| 8/8 [00:01<00:00,  5.36it/s, loss=0.793]


...


Epoch [298/300]
Train Loss: 0.0001  |  Train Acc: 1.0000
Val Loss:   1.1609  |  Val Acc:   0.8033


Epoch 299/300 [Train]: 100%|██████████| 8/8 [00:01<00:00,  4.93it/s, loss=0.000701]



Epoch [299/300]
Train Loss: 0.0002  |  Train Acc: 1.0000
Val Loss:   1.1746  |  Val Acc:   0.8033


Epoch 300/300 [Train]: 100%|██████████| 8/8 [00:01<00:00,  5.31it/s, loss=0.000209]



Epoch [300/300]
Train Loss: 0.0001  |  Train Acc: 1.0000
Val Loss:   1.1317  |  Val Acc:   0.8033

Training Complete ✅


In [4]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os

# ======== Parameters ========
num_classes = len(os.listdir('/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/Patch Classification/Sorted_data_split/test'))  # Same as training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======== Define Transforms (same as training) ========
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ======== Load Model ========
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load("resnet18_coconut_64x64.pth", map_location=device))
model = model.to(device)
model.eval()

# ======== Load Image and Preprocess ========
img_path = '/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/Patch Classification/Sorted_data_split/test/10/TR-1-D-16.png'
image = Image.open(img_path).convert("RGB")
image = transform(image).unsqueeze(0).to(device)  # Add batch dimension

# ======== Forward Pass + Softmax ========
with torch.no_grad():
    outputs = model(image)
    probs = torch.softmax(outputs, dim=1)  # Get probability distribution
    probs = probs.cpu().numpy().flatten()

# ======== Print Probabilities ========
for i, prob in enumerate(probs):
    print(f"Class {i}: {prob:.4f}")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


FileNotFoundError: [Errno 2] No such file or directory: 'resnet18_coconut_64x64.pth'

In [ ]:
import torch
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import os

# ======== Parameters ========
test_dir = '/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/patches_split/test'
batch_size = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======== Transforms ========
test_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ======== Load Test Dataset ========
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class_names = test_dataset.classes
num_classes = len(class_names)

# ======== Load Model ========
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load("resnet18_density_classifier.pth", map_location=device))
model.to(device)
model.eval()

# ======== Inference and Collect Predictions ========
all_preds = []
all_labels = []
all_paths = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Get paths from dataset.samples (keeps order with shuffle=False)
all_paths = [s[0] for s in test_dataset.samples]

# ======== Plot Misclassified Images ========
misclassified_indices = [i for i, (pred, true) in enumerate(zip(all_preds, all_labels)) if pred != true]

if not misclassified_indices:
    print("No misclassifications found.")
else:
    print(f"Found {len(misclassified_indices)} misclassified images.")

    # Show up to 25 misclassified images
    num_to_show = min(25, len(misclassified_indices))
    plt.figure(figsize=(15, 15))

    for i, idx in enumerate(misclassified_indices[:num_to_show]):
        image_path = all_paths[idx]
        image = plt.imread(image_path)

        plt.subplot(10, 10, i + 1)
        plt.imshow(image)
        plt.axis('off')
        plt.title(f"True: {class_names[all_labels[idx]]}\nPred: {class_names[all_preds[idx]]}", fontsize=9)

    plt.tight_layout()
    plt.suptitle("Misclassified Images", fontsize=16)
    plt.subplots_adjust(top=0.92)
    plt.show()

In [ ]:
import json
import numpy as np
import re
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# ===================== EDIT THESE =====================
IMAGES_ROOT = Path(r"/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/Patch_Data/Original")      # same as in csv_class_summarize.py
JSONS_DIR   = Path(r"/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/annotations")      # same as in csv_class_summarize.py
# =====================================================

# REQUIRED (already produced by your ResNet-18 inference)
# all_labels : list[int]  (true labels)
# all_preds  : list[int]  (predicted labels)
# class_names: list[str]  (from ImageFolder used for model)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}

# ---------- helpers (same logic as csv file) ----------
def natural_key(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", s)]

def normalize_stem(name):
    return Path(name).stem.lower()

def extract_pairs_from_json(obj):
    def yield_from_list(lst):
        for it in lst:
            if isinstance(it, dict):
                box_id = it.get("Box ID") or it.get("box_id") or it.get("boxId")
                pts = it.get("Points") or it.get("points")
                if isinstance(box_id, str) and isinstance(pts, list):
                    yield box_id, len(pts)

    if isinstance(obj, list):
        yield from yield_from_list(obj)
    elif isinstance(obj, dict):
        for k in ("boxes", "items", "data"):
            if k in obj and isinstance(obj[k], list):
                yield from yield_from_list(obj[k])
                return
        box_id = obj.get("Box ID") or obj.get("box_id") or obj.get("boxId")
        pts = obj.get("Points") or obj.get("points")
        if isinstance(box_id, str) and isinstance(pts, list):
            yield box_id, len(pts)

def build_box_index(jsons_root):
    index = {}
    for jp in jsons_root.rglob("*.json"):
        try:
            with jp.open("r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        for box_id, cnt in extract_pairs_from_json(data):
            key = normalize_stem(box_id)
            cnt = int(cnt)
            index[key] = max(index.get(key, 0), cnt)
    return index

# ---------- compute mean & sd per class (folder-based) ----------
box_index = build_box_index(JSONS_DIR)

per_class_counts = defaultdict(list)

for class_dir in sorted([p for p in IMAGES_ROOT.iterdir() if p.is_dir()],
                        key=lambda p: p.name.lower()):
    cls = class_dir.name
    for p in class_dir.iterdir():
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            key = p.stem.lower()
            if key in box_index:
                per_class_counts[cls].append(box_index[key])

class_stats = {}
for cls in class_names:
    counts = per_class_counts.get(cls, [])
    if len(counts) > 1:
        mu, sd = np.mean(counts), np.std(counts, ddof=1)
    elif len(counts) == 1:
        mu, sd = counts[0], 0.0
    else:
        mu, sd = 0.0, 0.0
    class_stats[cls] = (mu, sd)

# ---------- confusion matrix ----------
cm = confusion_matrix(all_labels, all_preds)

y_labels = [
    f"{cls}({class_stats[cls][0]:.0f},{class_stats[cls][1]:.1f})"
    for cls in class_names
]

fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

disp.plot(cmap=plt.cm.Blues, colorbar=False, ax=ax)

# vertical: class(mean,sd)
ax.set_yticklabels(y_labels)

# horizontal: class only
ax.set_xticklabels(class_names)

ax.set_title("Confusion Matrix (ResNet-18)")
plt.tight_layout()
plt.show()


In [ ]:
# ===================== IMPORTS =====================
import torch
import json
import re
import numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

# ===================== PATHS =====================
TEST_DIR = "/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/patches_split/test"
IMAGES_ROOT = Path("/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/Patch_Data/Original")
JSONS_DIR = Path("/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/annotations")
MODEL_PATH = "resnet18_density_classifier.pth"

# ===================== SETTINGS =====================
batch_size = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}

# ===================== JSON → BOX COUNT HELPERS =====================
def normalize_stem(name):
    return Path(name).stem.lower()

def extract_pairs_from_json(obj):
    def from_list(lst):
        for it in lst:
            if isinstance(it, dict):
                bid = it.get("Box ID") or it.get("box_id") or it.get("boxId")
                pts = it.get("Points") or it.get("points")
                if isinstance(bid, str) and isinstance(pts, list):
                    yield bid, len(pts)

    if isinstance(obj, list):
        yield from from_list(obj)
    elif isinstance(obj, dict):
        for k in ("boxes", "items", "data"):
            if k in obj and isinstance(obj[k], list):
                yield from from_list(obj[k])
                return
        bid = obj.get("Box ID") or obj.get("box_id") or obj.get("boxId")
        pts = obj.get("Points") or obj.get("points")
        if isinstance(bid, str) and isinstance(pts, list):
            yield bid, len(pts)

def build_box_index(json_root):
    index = {}
    for jp in json_root.rglob("*.json"):
        try:
            with open(jp, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        for box_id, cnt in extract_pairs_from_json(data):
            key = normalize_stem(box_id)
            index[key] = max(index.get(key, 0), cnt)
    return index

# ===================== COMPUTE MEAN & SD PER CLASS =====================
box_index = build_box_index(JSONS_DIR)
per_class_counts = defaultdict(list)

for class_dir in IMAGES_ROOT.iterdir():
    if not class_dir.is_dir():
        continue
    cls = class_dir.name
    for img in class_dir.iterdir():
        if img.suffix.lower() in IMAGE_EXTS:
            key = img.stem.lower()
            if key in box_index:
                per_class_counts[cls].append(box_index[key])

class_stats = {}
for cls, values in per_class_counts.items():
    if len(values) > 1:
        class_stats[cls] = (np.mean(values), np.std(values, ddof=1))
    elif len(values) == 1:
        class_stats[cls] = (values[0], 0.0)
    else:
        class_stats[cls] = (0.0, 0.0)

# ===================== DATASET & TRANSFORMS =====================
test_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
class_names = test_dataset.classes
num_classes = len(class_names)

# ===================== LOAD MODEL =====================
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

# ===================== INFERENCE =====================
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Image paths (order preserved)
all_paths = [s[0] for s in test_dataset.samples]

# ===================== MISCLASSIFIED IMAGES =====================
mis_idx = [i for i, (p, t) in enumerate(zip(all_preds, all_labels)) if p != t]

if not mis_idx:
    print("✅ No misclassifications found.")
else:
    print(f"❌ Found {len(mis_idx)} misclassified images")

    num_show = min(25, len(mis_idx))
    plt.figure(figsize=(15, 15))

    for i, idx in enumerate(mis_idx[:num_show]):
        img = plt.imread(all_paths[idx])

        true_cls = class_names[all_labels[idx]]
        pred_cls = class_names[all_preds[idx]]

        t_mu, t_sd = class_stats.get(true_cls, (0, 0))
        p_mu, p_sd = class_stats.get(pred_cls, (0, 0))

        plt.subplot(5, 5, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"True: {true_cls} ({t_mu:.0f}, {t_sd:.1f})\n"
            f"Pred: {pred_cls} ({p_mu:.0f}, {p_sd:.1f})",
            fontsize=9
        )

    plt.suptitle("Misclassified Images with (Mean, SD)", fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()


In [ ]:
# ===================== IMPORTS =====================
import torch
import json
import random
import math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

# ===================== PATHS =====================
TEST_DIR = "/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/patches_split/test"
IMAGES_ROOT = Path("/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/Patch_Data/Original")
JSONS_DIR = Path("/content/drive/Othercomputers/My Laptop/Project/Coconut Plantation/Applicatno/annotations")
MODEL_PATH = "resnet18_density_classifier.pth"

# ===================== SETTINGS =====================
batch_size = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}

# ===================== JSON → BOX COUNT =====================
def normalize_stem(name):
    return Path(name).stem.lower()

def extract_pairs_from_json(obj):
    def from_list(lst):
        for it in lst:
            if isinstance(it, dict):
                bid = it.get("Box ID") or it.get("box_id") or it.get("boxId")
                pts = it.get("Points") or it.get("points")
                if isinstance(bid, str) and isinstance(pts, list):
                    yield bid, len(pts)

    if isinstance(obj, list):
        yield from from_list(obj)
    elif isinstance(obj, dict):
        for k in ("boxes", "items", "data"):
            if k in obj and isinstance(obj[k], list):
                yield from from_list(obj[k])
                return
        bid = obj.get("Box ID") or obj.get("box_id") or obj.get("boxId")
        pts = obj.get("Points") or obj.get("points")
        if isinstance(bid, str) and isinstance(pts, list):
            yield bid, len(pts)

def build_box_index(json_root):
    index = {}
    for jp in json_root.rglob("*.json"):
        try:
            with open(jp, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        for box_id, cnt in extract_pairs_from_json(data):
            key = normalize_stem(box_id)
            index[key] = max(index.get(key, 0), cnt)
    return index

# ===================== MEAN & SD =====================
box_index = build_box_index(JSONS_DIR)
per_class_counts = defaultdict(list)

for class_dir in IMAGES_ROOT.iterdir():
    if class_dir.is_dir():
        for img in class_dir.iterdir():
            if img.suffix.lower() in IMAGE_EXTS:
                key = img.stem.lower()
                if key in box_index:
                    per_class_counts[class_dir.name].append(box_index[key])

class_stats = {
    cls: (np.mean(v), np.std(v, ddof=1)) if len(v) > 1 else
         (v[0], 0.0) if len(v) == 1 else (0.0, 0.0)
    for cls, v in per_class_counts.items()
}

# ===================== DATASET =====================
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(TEST_DIR, transform=transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
class_names = dataset.classes

# ===================== MODEL =====================
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

# ===================== INFERENCE =====================
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_paths = [p[0] for p in dataset.samples]

# ===================== CLASS → IMAGE LIST =====================
class_to_images = defaultdict(list)
for path, (_, label) in zip(all_paths, dataset.samples):
    class_to_images[class_names[label]].append(path)

# ===================== MISCLASSIFICATIONS =====================
mis_idx = [i for i, (p, t) in enumerate(zip(all_preds, all_labels)) if p != t]
print(f"Total misclassified patches: {len(mis_idx)}")

# ===================== PLOT (ONE STRAIGHT LINE) =====================
sets_per_row = 2
cols = sets_per_row * 2
rows = math.ceil(len(mis_idx) / sets_per_row)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = np.atleast_2d(axes)

for i, idx in enumerate(mis_idx):
    r = i // sets_per_row
    c = (i % sets_per_row) * 2

    true_cls = class_names[all_labels[idx]]
    pred_cls = class_names[all_preds[idx]]

    t_mu, t_sd = class_stats.get(true_cls, (0, 0))
    p_mu, p_sd = class_stats.get(pred_cls, (0, 0))

    mis_img = plt.imread(all_paths[idx])
    pred_img = plt.imread(random.choice(class_to_images[pred_cls]))

    axes[r, c].imshow(mis_img)
    axes[r, c].set_title(
        f"Mis Patch\nTrue: {true_cls} ({t_mu:.0f},{t_sd:.1f})",
        fontsize=9
    )
    axes[r, c].axis("off")

    axes[r, c + 1].imshow(pred_img)
    axes[r, c + 1].set_title(
        f"Random Pred\nPred: {pred_cls} ({p_mu:.0f},{p_sd:.1f})",
        fontsize=9
    )
    axes[r, c + 1].axis("off")

# ===================== ONE GLOBAL VERTICAL LINE =====================
x = axes[0, 1].get_position().x1
y_bottom = axes[-1, 0].get_position().y0
y_top = axes[0, 0].get_position().y1

fig.add_artist(
    plt.Line2D(
        [x, x],
        [y_bottom, y_top],
        transform=fig.transFigure,
        color="black",
        linewidth=3
    )
)

# ===================== CLEANUP =====================
for j in range(len(mis_idx) * 2, rows * cols):
    axes.flatten()[j].axis("off")

plt.suptitle("Misclassification Analysis (Single Divider)", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.show()
